##In this notebook, we simulate rainfall-runoff accross 222 CAMELS-AUS catchments using GR4J model.

In [1]:
from google.colab import files

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install git+https://github.com/kratzert/RRMPG.git

  Cloning https://github.com/kratzert/RRMPG.git to /tmp/pip-req-build-z5g9jux3
  Running command git clone --filter=blob:none --quiet https://github.com/kratzert/RRMPG.git /tmp/pip-req-build-z5g9jux3
  Resolved https://github.com/kratzert/RRMPG.git to commit 7de78c25acc1c255d2acaf739d65e9ce7bbd60c3
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.0 MB/s eta 0:00:00
  Created wheel for rrmpg: filename=rrmpg-0.1.1-py3-none-any.whl size=609986 sha256=03d14ecbcd3d96c76eba2da915dd70c72940ec77c670f0be6faf5554c229d031
  Stored in directory: /tmp/pip-ephem-wheel-cache-8gntk042/wheels/7d/ec/25/408ea8f9d8a1aff931ffd2c082074611b43cc13f5ec46e46b3
Successfully built rrmpg


In [7]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import random
from datetime import datetime, timedelta
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from rrmpg.models import GR4J
import zipfile
import os
from scipy.optimize import minimize

##CAMELS-DATA from my Google Drive.

In [5]:
# ==============================
# Paths to ZIP files
# ==============================
hydro_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/05_hydrometeorology.zip"
streamflow_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/03_streamflow.zip"

# Data extraction directories
hydro_dir = "/content/05_hydro"
streamflow_dir = "/content/03_streamflow"

# ==============================
# Function to extract ZIP files
# ==============================
def extract_zip(zip_path, extract_to):
    if not os.path.exists(extract_to):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✅ ZIP extracted in {extract_to}")
    else:
        print(f"✅ Directory {extract_to} already exists")

def find_csv(base_dir, csv_name):
    # Recursive search for the CSV file
    for root, dirs, files in os.walk(base_dir):
        if csv_name in files:
            return os.path.join(root, csv_name)
    raise FileNotFoundError(f"{csv_name} not found in {base_dir}")

# ==============================
# Extraction
# ==============================
extract_zip(hydro_zip, hydro_dir)
extract_zip(streamflow_zip, streamflow_dir)

# ==============================
# Load the 222 basins data
# ==============================
file_path = '/content/drive/MyDrive/Colab Notebooks/Dimension/id_name_metadata.csv'
basin222 = pd.read_csv(file_path)
station_ids_v1 = basin222['station_id'].astype(str).str.strip().unique()
print(f"✅ {len(station_ids_v1)} official basins loaded")

# ==============================
# 1️⃣ Precipitation (SILO)
# ==============================
precip_file = find_csv(hydro_dir, "precipitation_SILO.csv")
precip = pd.read_csv(precip_file, index_col=0, parse_dates=True)
precip.columns = precip.columns.str.strip()
precip.replace(-99.99, np.nan, inplace=True)
print("✅ SILO precipitation:", precip.shape)

# ==============================
# 2️⃣ Evapotranspiration (ET SILO)
# ==============================
et_file = find_csv(hydro_dir, "et_morton_actual_SILO.csv")
et = pd.read_csv(et_file, index_col=0, parse_dates=True)
et.columns = et.columns.str.strip()
et.replace(-99.99, np.nan, inplace=True)
print("✅ SILO ET:", et.shape)

# ==============================
# 3️⃣ Streamflow
# ==============================
streamflow_file = find_csv(streamflow_dir, "streamflow_mmd.csv")
Q = pd.read_csv(streamflow_file, index_col=0, parse_dates=True)
Q.columns = Q.columns.str.strip()
Q.replace(-99.99, np.nan, inplace=True)
print("✅ Streamflow:", Q.shape)

# ==============================
# 4️⃣ Identify common stations
# ==============================
stations_precip = set(precip.columns)
stations_et = set(et.columns)
stations_Q = set(Q.columns)

common_stations = [
    s for s in station_ids_v1
    if s in stations_precip and s in stations_et and s in stations_Q
]

print(f"✅ Official common stations: {len(common_stations)}")

# ==============================
# 5️⃣ Subset common stations
# ==============================
precip = precip[common_stations]
et = et[common_stations]
Q = Q[common_stations]

# ==============================
# 6️⃣ Final verification
# ==============================
print("Precipitation:", precip.shape)
print("ET:", et.shape)
print("Streamflow:", Q.shape)
print("Stations (first 10):", common_stations[:10], "...")


✅ ZIP extracted in /content/05_hydro
✅ ZIP extracted in /content/03_streamflow
✅ 222 official basins loaded
✅ SILO precipitation: (43464, 224)
✅ SILO ET: (43464, 224)
✅ Streamflow: (23376, 224)
✅ Official common stations: 222
Precipitation: (43464, 222)
ET: (43464, 222)
Streamflow: (23376, 222)
Stations (first 10): ['912101A', '912105A', '915011A', '917107A', '919003A', '919201A', '919309A', '922101B', '925001A', '926002A'] ...


GR4J

In [8]:
# ============================================
# General parameters
# ============================================
start_date = "1980-01-01"
end_date = "2014-12-31"
b1_ratio = 0.7
max_missing_ratio = 1.0

stations = common_stations  # 222 basins
results = {}

# ============================================
# Metric functions (NaNs ignored)
# ============================================
def NSE(obs, sim):
    df = pd.DataFrame({"obs": obs, "sim": sim}).dropna()
    if df.empty or df["obs"].var() == 0:
        return np.nan
    return 1 - np.sum((df["sim"] - df["obs"])**2) / np.sum(
        (df["obs"] - df["obs"].mean())**2
    )

def NNSE(obs, sim):
    nse = NSE(obs, sim)
    return 1 / (2 - nse) if not np.isnan(nse) else np.nan

# ============================================
# Runoff Ratio
# ============================================
def runoff_ratio(Q, P):
    df = pd.DataFrame({"Q": Q, "P": P}).dropna()
    if df.empty or df["P"].sum() == 0:
        return np.nan
    return df["Q"].sum() / df["P"].sum()

# ============================================
# GR4J objective function
# ============================================
def objective_gr4j(x, Q, P, ET):
    model = GR4J()
    params = dict(zip(model.get_parameter_names(), x))
    model.set_params(params)

    try:
        Qsim = model.simulate(P, ET).flatten()
        nse = NSE(Q, Qsim)
        return 1 - nse if np.isfinite(nse) else 1e6
    except Exception:
        return 1e6

# ============================================
# GR4J bounds
# ============================================
bounds = [
    (1, 3200),   # x1
    (-15, 15),   # x2
    (1, 1000),   # x3
    (0.5, 5)     # x4
]

# ============================================
# Main loop over basins
# ============================================
for i, station_id in enumerate(stations, start=1):

    print(f"\n=== Station {station_id} ({i}/{len(stations)}) ===")

    Q_obs = Q[station_id].loc[start_date:end_date].to_numpy(float)
    P = precip[station_id].loc[start_date:end_date].to_numpy(float)
    ET = et[station_id].loc[start_date:end_date].to_numpy(float)

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        print("⚠️ Station ignored (no valid data)")
        continue

    missing_ratio = np.isnan(Q_obs).sum() / N
    if missing_ratio > max_missing_ratio:
        print(f"⚠️ Too many missing values ({missing_ratio*100:.1f}%)")
        continue

    # ============================================
    # Runoff Ratio (entire period)
    # ============================================
    RR = runoff_ratio(Q_obs, P)

    # ============================================
    # Split calibration / validation
    # ============================================
    b1 = int(N * b1_ratio)

    Q_cal, Q_val = Q_obs[:b1], Q_obs[b1:]
    P_cal, P_val = P[:b1], P[b1:]
    ET_cal, ET_val = ET[:b1], ET[b1:]

    # ============================================
    # Multi-start calibration
    # ============================================
    best_fun = np.inf
    best_x = None
    np.random.seed(42)

    for _ in range(10):
        x0 = [np.random.uniform(b[0], b[1]) for b in bounds]

        res = minimize(
            objective_gr4j,
            x0,
            args=(Q_cal, P_cal, ET_cal),
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": 3000}
        )

        if res.fun < best_fun:
            best_fun = res.fun
            best_x = res.x

    if best_x is None:
        print("⚠️ Calibration failed")
        continue

    # ============================================
    # Final simulation
    # ============================================
    model = GR4J()
    params = dict(zip(model.get_parameter_names(), best_x))
    model.set_params(params)

    Qsim_cal = model.simulate(P_cal, ET_cal).flatten()
    Qsim_val = model.simulate(P_val, ET_val).flatten()

    # ============================================
    # Metrics
    # ============================================
    nse_cal = NSE(Q_cal, Qsim_cal)
    nnse_cal = NNSE(Q_cal, Qsim_cal)

    nse_val = NSE(Q_val, Qsim_val)
    nnse_val = NNSE(Q_val, Qsim_val)

    # ============================================
    # Store results
    # ============================================
    results[station_id] = {
        "RR": RR,
        "NSE_cal": nse_cal,
        "NNSE_cal": nnse_cal,
        "NSE_val": nse_val,
        "NNSE_val": nnse_val,
        **params
    }

    # ============================================
    # Display
    # ============================================
    print(f"Runoff Ratio (RR) : {RR:.3f}")
    print(f"NSE calibration  : {nse_cal:.3f}")
    print(f"NNSE calibration : {nnse_cal:.3f}")
    print(f"NSE validation   : {nse_val:.3f}")
    print(f"NNSE validation  : {nnse_val:.3f}")

print(f"\n✅ Finished : {len(results)} calibrated basins")



=== Station 912101A (1/222) ===
Runoff Ratio (RR) : 0.100
NSE calibration  : 0.788
NNSE calibration : 0.825
NSE validation   : 0.773
NNSE validation  : 0.815

=== Station 912105A (2/222) ===
Runoff Ratio (RR) : 0.118
NSE calibration  : 0.747
NNSE calibration : 0.798
NSE validation   : 0.640
NNSE validation  : 0.735

=== Station 915011A (3/222) ===
Runoff Ratio (RR) : 0.084
NSE calibration  : 0.641
NNSE calibration : 0.736
NSE validation   : 0.626
NNSE validation  : 0.728

=== Station 917107A (4/222) ===
Runoff Ratio (RR) : 0.115
NSE calibration  : 0.668
NNSE calibration : 0.751
NSE validation   : 0.597
NNSE validation  : 0.713

=== Station 919003A (5/222) ===
Runoff Ratio (RR) : 0.244
NSE calibration  : 0.749
NNSE calibration : 0.799
NSE validation   : 0.799
NNSE validation  : 0.833

=== Station 919201A (6/222) ===
Runoff Ratio (RR) : 0.228
NSE calibration  : 0.451
NNSE calibration : 0.645
NSE validation   : 0.367
NNSE validation  : 0.612

=== Station 919309A (7/222) ===
Runoff Ratio 

In [9]:
station_id = 803003  # or '803003' if needed

# Check if the station exists in Q
if station_id not in Q.columns:
    # Try as string
    station_id = str(station_id)
    if station_id not in Q.columns:
        raise ValueError(f"Station {station_id} not found in Q.columns.")

# Extract series
Q_obs = Q[station_id].loc[start_date:end_date]
N = len(Q_obs)
b1 = int(N * b1_ratio)

Q_cal = Q_obs[:b1]
Q_val = Q_obs[b1:]

# Count non-NaN values
non_nan_cal = Q_cal.notna().sum()
non_nan_val = Q_val.notna().sum()

print(f"Station {station_id}:")
print(f"  Number of non-NaN values in calibration : {non_nan_cal} / {len(Q_cal)}")
print(f"  Number of non-NaN values in validation  : {non_nan_val} / {len(Q_val)}")


Station 803003:
  Number of non-NaN values in calibration : 6735 / 8948
  Number of non-NaN values in validation  : 0 / 3836


In [10]:
# =============================================================
# 📌 EXTRACTION OF NSE & NNSE — CALIBRATION
# =============================================================
nse_cal = [res['NSE_cal'] for res in results.values()]
nnse_cal = [res['NNSE_cal'] for res in results.values()]

print("\n================= CALIBRATION =================\n")

if nse_cal:
    print(f"NSE Calibration -> Median: {np.percentile(nse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_cal, 95):.4f}")
    print("MEAN NSE_CAL:", np.mean(nse_cal))
    print("MIN NSE_CAL:", np.min(nse_cal))
    print("MAX NSE_CAL:", np.max(nse_cal))
else:
    print("No NSE available for calibration.")

print("\n--- NNSE Calibration ---")
if nnse_cal:
    print(f"NNSE Calibration -> Median: {np.percentile(nnse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(nnse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(nnse_cal, 95):.4f}")
    print("MEAN NNSE_CAL:", np.mean(nnse_cal))
    print("MIN NNSE_CAL:", np.min(nnse_cal))
    print("MAX NNSE_CAL:", np.max(nnse_cal))
else:
    print("No NNSE available for calibration.")


# =============================================================
# 📌 EXTRACTION OF NSE & NNSE — VALIDATION
# =============================================================
print("\n\n================= VALIDATION =================\n")

nse_val = [res['NSE_val'] for res in results.values() if not np.isnan(res['NSE_val'])]
nnse_val = [res['NNSE_val'] for res in results.values() if not np.isnan(res['NNSE_val'])]

if nse_val:
    print(f"NSE Validation -> Median: {np.percentile(nse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_val, 95):.4f}")
    print("MEAN NSE_VAL:", np.mean(nse_val))
    print("MIN NSE_VAL:", np.min(nse_val))
    print("MAX NSE_VAL:", np.max(nse_val))
else:
    print("No valid station for validation.")

print("\n--- NNSE Validation ---")
if nnse_val:
    print(f"NNSE Validation -> Median: {np.percentile(nnse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(nnse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(nnse_val, 95):.4f}")
    print("MEAN NNSE_VAL:", np.mean(nnse_val))
    print("MIN NNSE_VAL:", np.min(nnse_val))
    print("MAX NNSE_VAL:", np.max(nnse_val))
else:
    print("No valid station for NNSE in validation.")



================= CALIBRATION =================

NSE Calibration -> Median: 0.758, 5th percentile: 0.4820, 95th percentile: 0.8832
MEAN NSE_CAL: 0.742028346422303
MIN NSE_CAL: 0.20217438633220786
MAX NSE_CAL: 0.916742788382422

--- NNSE Calibration ---
NNSE Calibration -> Median: 0.805, 5th percentile: 0.6588, 95th percentile: 0.8954
MEAN NNSE_CAL: 0.8017174264985549
MIN NNSE_CAL: 0.556227474120737
MAX NNSE_CAL: 0.9231417887417025


================= VALIDATION =================

NSE Validation -> Median: 0.688, 5th percentile: 0.1535, 95th percentile: 0.8662
MEAN NSE_VAL: 0.6107864481363064
MIN NSE_VAL: -1.6320237843103849
MAX NSE_VAL: 0.9189500667249553

--- NNSE Validation ---
NNSE Validation -> Median: 0.762, 5th percentile: 0.5416, 95th percentile: 0.8820
MEAN NNSE_VAL: 0.7433026863704452
MIN NNSE_VAL: 0.27532859347447
MAX NNSE_VAL: 0.9250266516094185
